# Learning to Rank (LETOR) with XG-Boost

The XG-Boost package contains loss functions specifically desiged for ranking tasks (*rank:ndcg*, *rank:pairwise*, *rank:map*). I briefly explore these here by comparing the performance of the different loss functions over a simulated ranking task.

Import required libraries and set initial options:






In [1]:
import xgboost as xgb
#from sklearn.datasets import load_svmlight_file
import numpy as np
import pandas as pd
import random
import metrics_eval 
import math

pd.set_option('max.columns',999)
pd.set_option('max.rows',999)

np.random.seed(69420)

Simulate the data
(I will explain it in a bit):

In [2]:
n_features = 50
n_useless_features = 20
non_useless_feature_mask = [0]*n_useless_features + [1]*(n_features-n_useless_features) 
random.shuffle(non_useless_feature_mask)
n_items_per_person = 50
n_customers_train = 1000
n_customers_valid = 500
n_customers_test = 1000

feature_probs = np.array( [ random.random() for i in range(n_features) ] )
feature_probs = feature_probs / sum(feature_probs)
feature_probs = feature_probs * non_useless_feature_mask

# create training data:
current_customer_id = 0
train_customer_data_genlist = []
for customer_i in range(n_customers_train):
    current_customer_id = current_customer_id +1
    for item_j in range(n_items_per_person):
        features_vec = np.random.randint(low=0, high=2, size=n_features)
        prob_buy_this_item = sum( features_vec * feature_probs )
        y = int( random.random() < prob_buy_this_item )
        customer_row_df = pd.DataFrame( features_vec.reshape(-1, len(features_vec)), index=[current_customer_id] )
        customer_row_df.columns = [ 'X' + str(i+1) for i in range(n_features) ]
        customer_row_df['prob_buy_this_item'] = prob_buy_this_item
        customer_row_df['y'] = y
        train_customer_data_genlist.append( customer_row_df )
        
train_customer_data = pd.concat( train_customer_data_genlist )

# create validation data:
valid_customer_data_genlist = []
for customer_i in range(n_customers_valid):
    current_customer_id = current_customer_id +1
    for item_j in range(n_items_per_person):
        features_vec = np.random.randint(low=0, high=2, size=n_features)
        prob_buy_this_item = sum( features_vec * feature_probs )
        y = int( random.random() < prob_buy_this_item )
        customer_row_df = pd.DataFrame( features_vec.reshape(-1, len(features_vec)), index=[current_customer_id] )
        customer_row_df.columns = [ 'X' + str(i+1) for i in range(n_features) ]
        customer_row_df['prob_buy_this_item'] = prob_buy_this_item
        customer_row_df['y'] = y
        valid_customer_data_genlist.append( customer_row_df )
        
valid_customer_data = pd.concat( valid_customer_data_genlist )

# create test (holdout) data:
test_customer_data_genlist = []
for customer_i in range(n_customers_test):
    current_customer_id = current_customer_id +1
    for item_j in range(n_items_per_person):
        features_vec = np.random.randint(low=0, high=2, size=n_features)
        prob_buy_this_item = sum( features_vec * feature_probs )
        y = int( random.random() < prob_buy_this_item )
        customer_row_df = pd.DataFrame( features_vec.reshape(-1, len(features_vec)), index=[current_customer_id] )
        customer_row_df.columns = [ 'X' + str(i+1) for i in range(n_features) ]
        customer_row_df['prob_buy_this_item'] = prob_buy_this_item
        customer_row_df['y'] = y
        test_customer_data_genlist.append( customer_row_df )
        
test_customer_data = pd.concat( test_customer_data_genlist )

For example, here is the training data for a single customer (customer ID = 5): 

In [3]:
train_customer_data.iloc[5]

X1                    1.000000
X2                    0.000000
X3                    0.000000
X4                    1.000000
X5                    0.000000
X6                    0.000000
X7                    1.000000
X8                    0.000000
X9                    0.000000
X10                   0.000000
X11                   0.000000
X12                   1.000000
X13                   0.000000
X14                   0.000000
X15                   0.000000
X16                   1.000000
X17                   0.000000
X18                   1.000000
X19                   1.000000
X20                   1.000000
X21                   1.000000
X22                   1.000000
X23                   1.000000
X24                   1.000000
X25                   1.000000
X26                   1.000000
X27                   1.000000
X28                   1.000000
X29                   1.000000
X30                   1.000000
X31                   0.000000
X32                   1.000000
X33     

Each row of the data is a customer/product combination. For a given row, the probability of that particular customer buying that particular product is a function of the features. I haven't specified what the features are, but in reality they'd be attributes of the customer, attributes of the product, and features of the buying context (e.g. is it Summertime etc.).   

The model, of course, has view of the features (X) and outcome (y), but not of the true purchase probability.

The goal of a ranking model, then, is to rank the products within each customer (using the available features) from highest probability of purchase to lowest probability of purchase.

As a benchmark, I also model *probability of purchase* directly, using an XG-Boost classifier. Customers can then be ranked based on these predicted probabilities. I expect this approach to be inferior for the ranking task, but the question is of degree. 

Here is another view of all of the simulated datasets:

In [4]:
print( '--training data (head)--' )
print( train_customer_data.head(50) )
print( '--validation data (head)--' )
print( valid_customer_data.head(50) )
print( '--test data (head)--' )
print( test_customer_data.head(50) )

--training data (head)--
   X1  X2  X3  X4  X5  X6  X7  X8  X9  X10  X11  X12  X13  X14  X15  X16  X17  \
1   1   0   1   1   0   0   0   1   1    1    1    1    1    0    0    0    1   
1   0   1   1   1   0   0   1   0   1    1    0    1    0    0    1    0    1   
1   1   0   1   1   1   0   0   1   1    1    1    0    1    1    0    0    1   
1   0   0   1   1   1   0   0   1   0    0    1    0    0    0    1    0    1   
1   1   0   0   1   1   0   1   0   0    1    1    1    1    1    0    0    0   
1   1   0   0   1   0   0   1   0   0    0    0    1    0    0    0    1    0   
1   0   0   1   1   0   1   0   0   1    0    1    1    0    0    0    1    1   
1   1   1   1   0   1   0   1   1   0    1    0    1    0    1    0    1    1   
1   1   0   1   1   0   1   0   0   1    1    0    0    1    1    1    0    0   
1   0   0   0   1   1   0   1   1   0    1    1    0    1    1    1    0    1   
1   0   0   1   0   1   0   0   0   1    0    0    1    1    1    1    1    1   
1  

First, I train an XG-Boost model on the training dataset using the *rank:ndcg* (Normalised Discounted Cumulative Gain) objective:

In [5]:
ndcg_params = { 
            'objective': 'rank:ndcg',
            'learning_rate': 0.1,
            'gamma': 1.0,
            'min_child_weight': 0.1,
            'max_depth': 6,
            'n_estimators': 200,
            'eval_metric':['logloss','map@1','map@5','map@10','ndcg@1','ndcg@5','ndcg@10','ndcg']
}
ndcg_model = xgb.sklearn.XGBRanker( **ndcg_params )
ndcg_model.fit(
            X = train_customer_data[ [ col for col in train_customer_data.columns if col.startswith('X') ] ],               # feature matrix
            y = train_customer_data['y'],               # labels
            qid = np.array( train_customer_data.index ),            # query ID (groups within which ranking is performed)
            verbose = True,
            eval_set = [ 
                        ( 
                            valid_customer_data[ [ col for col in valid_customer_data.columns if col.startswith('X') ] ],
                            valid_customer_data['y']
                        )
                       ],
            eval_qid = [ np.array(valid_customer_data.index) ],
            early_stopping_rounds = 20         # stop model training if more than 10 rounds pass without improvement in ndcg
                                               # "Note that if you specify more than one evaluation metric the last one in param['eval_metric'] is used for early stopping."         
)

[0]	validation_0-logloss:0.69195	validation_0-map@1:0.02302	validation_0-map@5:0.07087	validation_0-map@10:0.11764	validation_0-ndcg@1:0.34800	validation_0-ndcg@5:0.33677	validation_0-ndcg@10:0.33992	validation_0-ndcg:0.67408
[1]	validation_0-logloss:0.68677	validation_0-map@1:0.02193	validation_0-map@5:0.07006	validation_0-map@10:0.11633	validation_0-ndcg@1:0.33000	validation_0-ndcg@5:0.33355	validation_0-ndcg@10:0.33671	validation_0-ndcg:0.67193
[2]	validation_0-logloss:0.68216	validation_0-map@1:0.02198	validation_0-map@5:0.07046	validation_0-map@10:0.11745	validation_0-ndcg@1:0.32400	validation_0-ndcg@5:0.33439	validation_0-ndcg@10:0.33778	validation_0-ndcg:0.67338
[3]	validation_0-logloss:0.67851	validation_0-map@1:0.02198	validation_0-map@5:0.07046	validation_0-map@10:0.11761	validation_0-ndcg@1:0.32400	validation_0-ndcg@5:0.33439	validation_0-ndcg@10:0.33803	validation_0-ndcg:0.67331
[4]	validation_0-logloss:0.67569	validation_0-map@1:0.02318	validation_0-map@5:0.07219	validatio

XGBRanker(base_score=0.5, booster='gbtree', colsample_bylevel=1,
          colsample_bynode=1, colsample_bytree=1,
          eval_metric=['logloss', 'map@1', 'map@5', 'map@10', 'ndcg@1',
                       'ndcg@5', 'ndcg@10', 'ndcg'],
          gamma=1.0, gpu_id=-1, importance_type='gain',
          interaction_constraints='', learning_rate=0.1, max_delta_step=0,
          max_depth=6, min_child_weight=0.1, missing=nan,
          monotone_constraints='()', n_estimators=200, n_jobs=12,
          num_parallel_tree=1, objective='rank:ndcg', random_state=0,
          reg_alpha=0, reg_lambda=1, scale_pos_weight=None, subsample=1,
          tree_method='exact', validate_parameters=1, verbosity=None)

Extract test-set predictions from the trained (*rank:ndcg* objective) XG-Boost model:

In [6]:
ndcg_testset_preds = ndcg_model.predict( 
                                          test_customer_data[ [ col for col in test_customer_data.columns if col.startswith('X') ] ].copy(),
                                          iteration_range = (0, ndcg_model.best_iteration)
                                       )
test_customer_data['ndcg_testset_preds'] = ndcg_testset_preds

C:\Program Files (x86)\Microsoft Visual Studio\Shared\Python37_64\lib\site-packages\xgboost\data.py:114: UserWarning: Use subset (sliced data) of np.ndarray is not recommended because it will generate extra copies and increase memory consumption
  "because it will generate extra copies and increase " +


Now, I train an XG-Boost model on the same training data using the pairwise ranking objective:

In [7]:
rank_pairwise_params = { 
            'objective': 'rank:pairwise',
            'learning_rate': 0.1,
            'gamma': 1.0,
            'min_child_weight': 0.1,
            'max_depth': 6,
            'n_estimators': 200,
            'eval_metric':['logloss','map@1','map@5','map@10','ndcg@1','ndcg@5','ndcg@10','ndcg']
}
rank_pairwise_model = xgb.sklearn.XGBRanker( **rank_pairwise_params )
rank_pairwise_model.fit(
            X = train_customer_data[ [ col for col in train_customer_data.columns if col.startswith('X') ] ],               # feature matrix
            y = train_customer_data['y'],               # labels
            qid = np.array( train_customer_data.index ),            # query ID (groups within which ranking is performed)
            verbose = True,
            eval_set = [ 
                        ( 
                            valid_customer_data[ [ col for col in valid_customer_data.columns if col.startswith('X') ] ],
                            valid_customer_data['y']
                        )
                       ],
            eval_qid = [ np.array(valid_customer_data.index) ],
            early_stopping_rounds = 20         # stop model training if more than 10 rounds pass without improvement in ndcg
                                               # "Note that if you specify more than one evaluation metric the last one in param['eval_metric'] is used for early stopping."         
)

[0]	validation_0-logloss:0.69087	validation_0-map@1:0.02381	validation_0-map@5:0.07336	validation_0-map@10:0.12039	validation_0-ndcg@1:0.35600	validation_0-ndcg@5:0.34308	validation_0-ndcg@10:0.34252	validation_0-ndcg:0.67828
[1]	validation_0-logloss:0.68926	validation_0-map@1:0.02457	validation_0-map@5:0.07610	validation_0-map@10:0.12331	validation_0-ndcg@1:0.36200	validation_0-ndcg@5:0.35254	validation_0-ndcg@10:0.34781	validation_0-ndcg:0.68054
[2]	validation_0-logloss:0.68801	validation_0-map@1:0.02562	validation_0-map@5:0.07899	validation_0-map@10:0.12595	validation_0-ndcg@1:0.38000	validation_0-ndcg@5:0.36010	validation_0-ndcg@10:0.35136	validation_0-ndcg:0.68403
[3]	validation_0-logloss:0.68703	validation_0-map@1:0.02559	validation_0-map@5:0.07985	validation_0-map@10:0.12917	validation_0-ndcg@1:0.36800	validation_0-ndcg@5:0.36274	validation_0-ndcg@10:0.35710	validation_0-ndcg:0.68594
[4]	validation_0-logloss:0.68643	validation_0-map@1:0.02524	validation_0-map@5:0.08031	validatio

XGBRanker(base_score=0.5, booster='gbtree', colsample_bylevel=1,
          colsample_bynode=1, colsample_bytree=1,
          eval_metric=['logloss', 'map@1', 'map@5', 'map@10', 'ndcg@1',
                       'ndcg@5', 'ndcg@10', 'ndcg'],
          gamma=1.0, gpu_id=-1, importance_type='gain',
          interaction_constraints='', learning_rate=0.1, max_delta_step=0,
          max_depth=6, min_child_weight=0.1, missing=nan,
          monotone_constraints='()', n_estimators=200, n_jobs=12,
          num_parallel_tree=1, random_state=0, reg_alpha=0, reg_lambda=1,
          scale_pos_weight=None, subsample=1, tree_method='exact',
          validate_parameters=1, verbosity=None)

Get test set predictions from the XG-Boost model trained using *rank:pairwise* objective:

In [8]:
rank_pairwise_testset_preds = rank_pairwise_model.predict( 
    test_customer_data[ [ col for col in test_customer_data.columns if col.startswith('X') ] ].copy(),
    iteration_range = (0, rank_pairwise_model.best_iteration)                       
)
test_customer_data['rank_pairwise_testset_preds'] = rank_pairwise_testset_preds

C:\Program Files (x86)\Microsoft Visual Studio\Shared\Python37_64\lib\site-packages\xgboost\data.py:114: UserWarning: Use subset (sliced data) of np.ndarray is not recommended because it will generate extra copies and increase memory consumption
  "because it will generate extra copies and increase " +


Train XG-Boost model using the objective function *rank:map* (Mean Average Precision):

In [9]:
map_params = { 
            'objective': 'rank:map',
            'learning_rate': 0.1,
            'gamma': 1.0,
            'min_child_weight': 0.1,
            'max_depth': 6,
            'n_estimators': 200,
            'eval_metric':['logloss','map@1','map@5','map@10','ndcg@1','ndcg@5','ndcg@10','ndcg','map']
}
map_model = xgb.sklearn.XGBRanker( **map_params )
map_model.fit(
            X = train_customer_data[ [ col for col in train_customer_data.columns if col.startswith('X') ] ],               # feature matrix
            y = train_customer_data['y'],               # labels
            qid = np.array( train_customer_data.index ),            # query ID (groups within which ranking is performed)
            verbose = True,
            eval_set = [ 
                        ( 
                            valid_customer_data[ [ col for col in valid_customer_data.columns if col.startswith('X') ] ],
                            valid_customer_data['y']
                        )
                       ],
            eval_qid = [ np.array(valid_customer_data.index) ],
            early_stopping_rounds = 20      # stop model training if MAP does not improve in 10 consecutive rounds
)

[0]	validation_0-logloss:0.69214	validation_0-map@1:0.02106	validation_0-map@5:0.06738	validation_0-map@10:0.11287	validation_0-ndcg@1:0.31800	validation_0-ndcg@5:0.31852	validation_0-ndcg@10:0.32768	validation_0-ndcg:0.66801	validation_0-map:0.36727
[1]	validation_0-logloss:0.68935	validation_0-map@1:0.02119	validation_0-map@5:0.06884	validation_0-map@10:0.11265	validation_0-ndcg@1:0.32000	validation_0-ndcg@5:0.32809	validation_0-ndcg@10:0.32897	validation_0-ndcg:0.66952	validation_0-map:0.36844
[2]	validation_0-logloss:0.68662	validation_0-map@1:0.02119	validation_0-map@5:0.06928	validation_0-map@10:0.11369	validation_0-ndcg@1:0.32000	validation_0-ndcg@5:0.33002	validation_0-ndcg@10:0.33054	validation_0-ndcg:0.67035	validation_0-map:0.36968
[3]	validation_0-logloss:0.68451	validation_0-map@1:0.02186	validation_0-map@5:0.07368	validation_0-map@10:0.11776	validation_0-ndcg@1:0.33000	validation_0-ndcg@5:0.34508	validation_0-ndcg@10:0.33669	validation_0-ndcg:0.67545	validation_0-map:0.37

XGBRanker(base_score=0.5, booster='gbtree', colsample_bylevel=1,
          colsample_bynode=1, colsample_bytree=1,
          eval_metric=['logloss', 'map@1', 'map@5', 'map@10', 'ndcg@1',
                       'ndcg@5', 'ndcg@10', 'ndcg', 'map'],
          gamma=1.0, gpu_id=-1, importance_type='gain',
          interaction_constraints='', learning_rate=0.1, max_delta_step=0,
          max_depth=6, min_child_weight=0.1, missing=nan,
          monotone_constraints='()', n_estimators=200, n_jobs=12,
          num_parallel_tree=1, objective='rank:map', random_state=0,
          reg_alpha=0, reg_lambda=1, scale_pos_weight=None, subsample=1,
          tree_method='exact', validate_parameters=1, verbosity=None)

get test-set predictions from trained XG-Boost model trained using MAP ranking objective:

In [10]:
map_testset_preds = map_model.predict( 
                        test_customer_data[ [ col for col in test_customer_data.columns if col.startswith('X') ] ].copy(),
                        iteration_range = (0, map_model.best_iteration)                       
)
test_customer_data['map_testset_preds'] = map_testset_preds

Train an XG-Boost classifier, using logistic loss:

In [11]:
classifier_params = { 
            'objective': 'binary:logistic',
            'learning_rate': 0.1,
            'gamma': 1.0,
            'min_child_weight': 0.1,
            'max_depth': 6,
            'n_estimators': 100,
            'eval_metric':['map@1','map@5','map@10','ndcg@1','ndcg@5','ndcg@10','logloss']
}
classifier_model = xgb.XGBClassifier( **classifier_params )
classifier_model.fit(
            X = train_customer_data[ [ col for col in train_customer_data.columns if col.startswith('X') ] ],               # feature matrix
            y = train_customer_data['y'],               # labels
            verbose = True,
            eval_set = [ 
                        ( 
                            valid_customer_data[ [ col for col in valid_customer_data.columns if col.startswith('X') ] ],
                            valid_customer_data['y']
                        )
                       ],
            early_stopping_rounds = 20        # stop the model training if log-loss doesn't improve for 10 consecutive rounds
)

[0]	validation_0-map@1:0.00013	validation_0-map@5:0.00027	validation_0-map@10:0.00027	validation_0-ndcg@1:1.00000	validation_0-ndcg@5:0.55315	validation_0-ndcg@10:0.35895	validation_0-logloss:0.67716
[1]	validation_0-map@1:0.00013	validation_0-map@5:0.00027	validation_0-map@10:0.00056	validation_0-ndcg@1:1.00000	validation_0-ndcg@5:0.55315	validation_0-ndcg@10:0.63666	validation_0-logloss:0.66423
[2]	validation_0-map@1:0.00000	validation_0-map@5:0.00003	validation_0-map@10:0.00017	validation_0-ndcg@1:0.00000	validation_0-ndcg@5:0.13120	validation_0-ndcg@10:0.29342	validation_0-logloss:0.65361


C:\Program Files (x86)\Microsoft Visual Studio\Shared\Python37_64\lib\site-packages\xgboost\sklearn.py:1146: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)


[3]	validation_0-map@1:0.00000	validation_0-map@5:0.00003	validation_0-map@10:0.00017	validation_0-ndcg@1:0.00000	validation_0-ndcg@5:0.13120	validation_0-ndcg@10:0.29342	validation_0-logloss:0.64494
[4]	validation_0-map@1:0.00000	validation_0-map@5:0.00011	validation_0-map@10:0.00026	validation_0-ndcg@1:0.00000	validation_0-ndcg@5:0.31565	validation_0-ndcg@10:0.35660	validation_0-logloss:0.63776
[5]	validation_0-map@1:0.00000	validation_0-map@5:0.00011	validation_0-map@10:0.00026	validation_0-ndcg@1:0.00000	validation_0-ndcg@5:0.31565	validation_0-ndcg@10:0.35660	validation_0-logloss:0.63182
[6]	validation_0-map@1:0.00000	validation_0-map@5:0.00009	validation_0-map@10:0.00029	validation_0-ndcg@1:0.00000	validation_0-ndcg@5:0.27727	validation_0-ndcg@10:0.39138	validation_0-logloss:0.62691
[7]	validation_0-map@1:0.00000	validation_0-map@5:0.00009	validation_0-map@10:0.00029	validation_0-ndcg@1:0.00000	validation_0-ndcg@5:0.27727	validation_0-ndcg@10:0.39138	validation_0-logloss:0.62295


XGBClassifier(base_score=0.5, booster='gbtree', colsample_bylevel=1,
              colsample_bynode=1, colsample_bytree=1,
              eval_metric=['map@1', 'map@5', 'map@10', 'ndcg@1', 'ndcg@5',
                           'ndcg@10', 'logloss'],
              gamma=1.0, gpu_id=-1, importance_type='gain',
              interaction_constraints='', learning_rate=0.1, max_delta_step=0,
              max_depth=6, min_child_weight=0.1, missing=nan,
              monotone_constraints='()', n_estimators=100, n_jobs=12,
              num_parallel_tree=1, random_state=0, reg_alpha=0, reg_lambda=1,
              scale_pos_weight=1, subsample=1, tree_method='exact',
              validate_parameters=1, verbosity=None)

get test-set predictions from trained XG-Boost classifier model trained with logistic loss

In [12]:
classifier_testset_preds = classifier_model.predict_proba( 
                        test_customer_data[ [ col for col in test_customer_data.columns if col.startswith('X') ] ].copy(),
                        iteration_range = (0, classifier_model.best_iteration)                       
)
test_customer_data['classifier_testset_preds'] = classifier_testset_preds[:,1]

Now, I calculate the predicted (and true) rank of each item within each customer on the test set:

In [13]:
test_customer_data['true_item_rank'] = test_customer_data.groupby(test_customer_data.index)['prob_buy_this_item'].rank(method='dense', ascending=False).astype(int)
test_customer_data['ndcg_item_rank'] = test_customer_data.groupby(test_customer_data.index)['ndcg_testset_preds'].rank(method='dense', ascending=False).astype(int)
test_customer_data['rank_pairwise_item_rank'] = test_customer_data.groupby(test_customer_data.index)['rank_pairwise_testset_preds'].rank(method='dense', ascending=False).astype(int)
test_customer_data['map_item_rank'] = test_customer_data.groupby(test_customer_data.index)['map_testset_preds'].rank(method='dense', ascending=False).astype(int)
test_customer_data['classifier_item_rank'] = test_customer_data.groupby(test_customer_data.index)['classifier_testset_preds'].rank(method='dense', ascending=False).astype(int)

In [14]:
test_customer_data.head(50)

,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,X11,X12,X13,X14,X15,X16,X17,X18,X19,X20,X21,X22,X23,X24,X25,X26,X27,X28,X29,X30,X31,X32,X33,X34,X35,X36,X37,X38,X39,X40,X41,X42,X43,X44,X45,X46,X47,X48,X49,X50,prob_buy_this_item,y,ndcg_testset_preds,rank_pairwise_testset_preds,map_testset_preds,classifier_testset_preds,true_item_rank,ndcg_item_rank,rank_pairwise_item_rank,map_item_rank,classifier_item_rank
1501,1,1,0,1,0,1,0,1,0,0,1,0,1,1,0,1,0,0,1,1,0,0,1,0,1,0,1,0,1,1,1,0,1,0,0,0,0,0,0,1,0,0,1,0,0,1,1,0,1,0,0.290265,0,0.379513,0.400322,0.317998,0.291919,31,29,37,33,28
1501,0,1,1,0,1,1,1,0,0,0,1,1,0,0,1,1,0,1,1,1,1,0,1,0,0,1,1,1,1,1,0,0,0,0,1,1,0,1,0,1,0,0,0,1,1,0,0,1,1,1,0.301621,0,0.504727,0.642729,0.566262,0.300164,28,15,11,13,25
1501,1,0,1,1,1,1,1,1,0,1,0,1,1,0,1,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,1,0,1,0,0,0,1,1,0,0,0.277391,0,0.275054,0.459391,0.150150,0.250917,35,39,30,45,45
1501,0,1,1,0,0,0,0,1,1,0,0,0,0,1,1,1,1,0,0,1,1,1,1,0,0,1,1,0,0,1,0,0,0,1,0,1,1,1,1,0,1,0,0,0,0,0,0,0,1,1,0.309641,0,0.592828,0.709741,0.770189,0.368310,24,6,3,3,5
1501,0,1,0,1,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,1,0,0,0,0,1,0,1,1,0,0,0,1,1,0,0,0,0,0,1,0,0,1,0,1,0,1,1,1,1,0.238565,0,0.214200,0.360356,0.225156,0.260784,43,46,43,41,42
1501,1,1,0,1,0,1,1,0,1,0,0,1,0,0,1,0,0,1,1,1,1,0,0,1,0,0,0,1,0,1,1,0,1,1,0,0,0,0,1,1,0,0,1,0,0,1,1,1,1,1,0.287414,0,0.362592,0.565024,0.362687,0.352758,32,30,16,31,9
1501,1,0,1,1,1,1,1,0,0,1,1,0,0,0,1,0,0,1,1,0,1,0,1,0,1,1,1,0,0,1,0,1,1,0,0,1,0,1,0,0,1,1,0,1,1,1,1,1,1,1,0.360258,0,0.606351,0.668506,0.713943,0.331735,7,4,7,4,15
1501,1,1,0,0,0,1,1,0,0,1,0,0,1,1,0,0,1,1,0,0,0,1,1,1,1,1,1,0,1,0,0,0,0,0,0,1,1,0,0,1,1,1,0,0,1,1,0,0,0,0,0.378827,0,0.737380,0.647413,0.776575,0.335128,4,1,9,2,13
1501,1,1,1,1,1,0,1,0,1,0,0,1,0,1,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0,0,0,1,0,1,1,1,1,0,0,0,0,0,0,1,0,1,1,1,1,0,0.249626,0,0.325056,0.475770,0.264675,0.292751,41,33,27,39,27
1501,0,0,0,0,0,0,1,0,1,1,1,0,1,0,1,0,0,0,1,1,1,0,0,1,0,1,0,0,0,0,1,1,0,0,1,0,0,1,1,0,0,0,0,1,1,0,1,0,1,1,0.236146,0,0.150853,0.392873,0.100594,0.275848,45,49,38,47,34


Compare the performance of the different models on the test set using various metrics:

In [15]:
# make a random number column for breaking ties:
test_customer_data['runif01'] = np.random.uniform(size=len(test_customer_data))

print( '-- AVERAGE (MEAN) TRUE RANK OF TOP-RATED ITEM --' )
print('ndcg:')
print(
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','ndcg_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(1)
    .agg( ndcg = ('true_item_rank','mean') )
    .values
    [0]
)
print('rank pairwise:')
print(
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','rank_pairwise_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(1)
    .agg( rank_pairwise = ('true_item_rank','mean') )
    .values
    [0]
)
print('MAP:')
print(
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','map_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(1)
    .agg( map_rank = ('true_item_rank','mean') )
    .values
    [0]
)
print('Classifier:')
print(
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values( ['index','classifier_item_rank','runif01'] )   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(1)
    .agg( classifier = ('true_item_rank','mean') )
    .values
    [0]    
)


print( '\n-- AVERAGE (MEAN) TRUE RANK OF TOP-RATED 2 ITEMS --' )
print('ndcg:')
print(
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','ndcg_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(2)
    .agg( ndcg_rank = ('true_item_rank','mean') )
    .values
    [0]    
)
print('rank pairwise:')
print(
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','rank_pairwise_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(2)
    .agg( rank_pairwise = ('true_item_rank','mean') )
    .values
    [0]    
)
print('MAP:')
print(
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','map_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(2)
    .agg( map_rank = ('true_item_rank','mean') )
    .values
    [0]    
)
print('Classifier:')
print(
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values( ['index','classifier_item_rank','runif01'] )   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(2)
    .agg( classifier = ('true_item_rank','mean') )
    .values
    [0]    
)

print( '\n-- AVERAGE (MEAN) TRUE RANK OF TOP-RATED 5 ITEMS --' )
print('ndcg:')
print(
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','ndcg_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(5)
    .agg( ndcg = ('true_item_rank','mean') )
    .values
    [0]    
)
print('rank pairwise:')
print(
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','rank_pairwise_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(5)
    .agg( rank_pairwise = ('true_item_rank','mean') )
    .values
    [0]    
)
print('MAP:')
print(
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','map_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(5)
    .agg( map_rank = ('true_item_rank','mean') )
    .values
    [0]    
)
print('Classifier:')
print(
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values( ['index','classifier_item_rank','runif01'] )   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(5)
    .agg( classifier = ('true_item_rank','mean') )
    .values
    [0]    
)

print( '\n-- AVERAGE (MEAN) TRUE RANK OF 10 TOP-RATED ITEMS --' )
print('ndcg:')
print(
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','ndcg_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(10)
    .agg( ndcg_rank = ('true_item_rank','mean') )
    .values
    [0]    
)
print('rank pairwise:')
print(
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','rank_pairwise_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(10)
    .agg( rank_pairwise = ('true_item_rank','mean') )
    .values
    [0]    
)
print('MAP:')
print(
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','map_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(10)
    .agg( map_rank = ('true_item_rank','mean') )
    .values
    [0]    
)
print('Classifier:')
print(
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values( ['index','classifier_item_rank','runif01'] )   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(10)
    .agg( classifier = ('true_item_rank','mean') )
    .values
    [0]    
)

print( '\n-- SUM PROBABILITY OF TOP RATED ITEM --' )
print('true item rank:')
print( 
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','true_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(1)
    ['prob_buy_this_item']
    .sum() 
)
print('ndcg:')
print( 
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','ndcg_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(1)
    ['prob_buy_this_item']
    .sum() 
)
print('rank pairwise:')
print( 
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','rank_pairwise_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(1)
    ['prob_buy_this_item']
    .sum() 
)
print('MAP:')
print( 
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','map_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(1)
    ['prob_buy_this_item']
    .sum() 
)
print('classifier:')
print( 
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','classifier_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(1)
    ['prob_buy_this_item']
    .sum() 
)

print( '\n-- SUM PROBABILITY OF TOP RATED 5 ITEMS --' )
print('true item rank:')
print( 
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','true_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(5)
    ['prob_buy_this_item']
    .sum() 
)
print('ndcg:')
print( 
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','ndcg_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(5)
    ['prob_buy_this_item']
    .sum() 
)
print('rank pairwise:')
print( 
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','rank_pairwise_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(5)
    ['prob_buy_this_item']
    .sum() 
)
print('MAP:')
print( 
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','map_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(5)
    ['prob_buy_this_item']
    .sum() 
)
print('classifier:')
print( 
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','classifier_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(5)
    ['prob_buy_this_item']
    .sum() 
)

print( '\n-- SUM PROBABILITY OF TOP RATED 10 ITEMS --' )
print('true item rank:')
print( 
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','true_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(10)
    ['prob_buy_this_item']
    .sum() 
)
print('ndcg:')
print( 
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','ndcg_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(10)
    ['prob_buy_this_item']
    .sum() 
)
print('rank pairwise:')
print( 
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','rank_pairwise_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(10)
    ['prob_buy_this_item']
    .sum() 
)
print('MAP:')
print( 
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','map_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(10)
    ['prob_buy_this_item']
    .sum() 
)
print('classifier:')
print( 
    test_customer_data
    .reset_index() # make customer ID into a column (it is named "index")
    .sort_values(['index','classifier_item_rank','runif01'])   # sort by customer ID, then NDCG predicted rank, settling ties randomly (runif01)
    .groupby('index')
    .head(10)
    ['prob_buy_this_item']
    .sum() 
)

print( '\n-- Discounted Cumulative Probability --' )
print( '...will do this later' )


-- AVERAGE (MEAN) TRUE RANK OF TOP-RATED ITEM --
ndcg:
[5.171]
rank pairwise:
[3.493]
MAP:
[4.032]
Classifier:
[4.113]

-- AVERAGE (MEAN) TRUE RANK OF TOP-RATED 2 ITEMS --
ndcg:
[6.4775]
rank pairwise:
[4.4335]
MAP:
[4.851]
Classifier:
[4.9455]

-- AVERAGE (MEAN) TRUE RANK OF TOP-RATED 5 ITEMS --
ndcg:
[8.8288]
rank pairwise:
[6.5144]
MAP:
[6.9214]
Classifier:
[6.942]

-- AVERAGE (MEAN) TRUE RANK OF 10 TOP-RATED ITEMS --
ndcg:
[11.3976]
rank pairwise:
[9.157]
MAP:
[9.4829]
Classifier:
[9.5268]

-- SUM PROBABILITY OF TOP RATED ITEM --
true item rank:
439.90412428669316
ndcg:
402.4043403616702
rank pairwise:
416.0995351725984
MAP:
411.66203894580275
classifier:
411.8742188489265

-- SUM PROBABILITY OF TOP RATED 5 ITEMS --
true item rank:
2039.7216496872575
ndcg:
1884.7948587466378
rank pairwise:
1945.7693427016536
MAP:
1935.3503377945626
classifier:
1934.9575474907795

-- SUM PROBABILITY OF TOP RATED 10 ITEMS --
true item rank:
3876.927926113749
ndcg:
3630.9162796149876
rank pairwise:
37

For validation, here is a replication of the nDCG@k and MAP@k calcs used by the XG-Boost package (I replicate the metrics on the validation set):

In [19]:
# replicating ndcg@k on the validation set for the MAP-objective model: 
metric_eval_map_validset_preds = pd.DataFrame(
 {   
    'User_ID':valid_customer_data.index,
    'y_observed_outcome':valid_customer_data['y'],
     'map_model_preds':map_model.predict( 
         valid_customer_data[ [ col for col in valid_customer_data.columns if col.startswith('X') ] ].copy(),
         iteration_range = (0, map_model.best_iteration)
     )
 }
)

metric_eval_map_validset_preds = (
    metric_eval_map_validset_preds
    .sort_values( ['User_ID', 'map_model_preds'], ascending=[True,False] )
)
metric_eval_map_validset_preds['item_rank'] = (
    metric_eval_map_validset_preds
    .groupby( 'User_ID' )
    .cumcount()+1
)
metric_eval_map_validset_preds['DG'] = metric_eval_map_validset_preds.apply( 
    lambda row: (2**row['y_observed_outcome'] -1) / np.log2( row['item_rank'] + 1 ),
    axis=1 
)
metric_eval_map_validset_preds['DCG'] = (
    metric_eval_map_validset_preds
    .groupby('User_ID')
    ['DG']
    .cumsum()
)

# calculate ideal NDCG:
ideal_ndcg_calc = metric_eval_map_validset_preds.copy()[['User_ID','y_observed_outcome']]
ideal_ndcg_calc = ideal_ndcg_calc.sort_values(['User_ID', 'y_observed_outcome'], ascending=[True, False])
ideal_ndcg_calc['item_rank'] = (
    ideal_ndcg_calc
    .groupby( 'User_ID' )
    .cumcount()+1
)
ideal_ndcg_calc['DG'] = ideal_ndcg_calc.apply( 
    lambda row: (2**row['y_observed_outcome'] -1) / np.log2( row['item_rank'] + 1 ),
    axis=1 
)
ideal_ndcg_calc['ideal_DCG'] = (
    ideal_ndcg_calc
    .groupby('User_ID')
    ['DG']
    .cumsum()
)
metric_eval_map_validset_preds = metric_eval_map_validset_preds.merge(
    ideal_ndcg_calc[['User_ID','item_rank','ideal_DCG']],
    how = 'left',
    on = ['User_ID', 'item_rank']
)
metric_eval_map_validset_preds['nDCG'] = metric_eval_map_validset_preds['DCG'] / metric_eval_map_validset_preds['ideal_DCG']

print( f"nDCG@1: {metric_eval_map_validset_preds.query('item_rank==1')['nDCG'].mean()}" )
print( f"nDCG@5: {metric_eval_map_validset_preds.query('item_rank==5')['nDCG'].mean()}" )
print( f"nDCG@10: {metric_eval_map_validset_preds.query('item_rank==10')['nDCG'].mean()}" )

# replicating map@k on the validation set for the MAP-objective model: 
total_relevant_items_per_user = (
    metric_eval_map_validset_preds
    [['User_ID', 'y_observed_outcome']]
    .groupby('User_ID')
    .agg( n_relevant_items_this_user = ('y_observed_outcome','sum') )
    .reset_index()
)

metric_eval_map_validset_preds = (
    metric_eval_map_validset_preds
    .merge( 
            total_relevant_items_per_user,
            how = 'left',
            on = 'User_ID'
          )
)

metric_eval_map_validset_preds['cum_n_relevant_items'] = (
    metric_eval_map_validset_preds
    .groupby('User_ID')
    ['y_observed_outcome']
    .cumsum()
)

metric_eval_map_validset_preds['cum_precision'] = (
    metric_eval_map_validset_preds['cum_n_relevant_items'] /
    metric_eval_map_validset_preds['item_rank']
)

metric_eval_map_validset_preds['cum_precision * y'] = (
    metric_eval_map_validset_preds['cum_precision'] *
    metric_eval_map_validset_preds['y_observed_outcome']
)

metric_eval_map_validset_preds['cumsum( cum_precision * y )'] = (
    metric_eval_map_validset_preds.groupby('User_ID')['cum_precision * y'].cumsum()
)

metric_eval_map_validset_preds['min{k,n_relevant_items}'] = (
    metric_eval_map_validset_preds[['item_rank','n_relevant_items_this_user']].min( axis=1 )
)

metric_eval_map_validset_preds['AP@k'] = (
    metric_eval_map_validset_preds['cumsum( cum_precision * y )'] /
    metric_eval_map_validset_preds['n_relevant_items_this_user']
)

print( f"MAP@1: {metric_eval_map_validset_preds.query('item_rank==1')['AP@k'].mean()}")
print( f"MAP@5: {metric_eval_map_validset_preds.query('item_rank==5')['AP@k'].mean()}")
print( f"MAP@10: {metric_eval_map_validset_preds.query('item_rank==10')['AP@k'].mean()}")

#metric_eval_map_validset_preds.head(50)

nDCG@1: 0.406
nDCG@5: 0.3776725179540275
nDCG@10: 0.36678168861108745
MAP@1: 0.02756995049154492
MAP@5: 0.08422812461789395
MAP@10: 0.1342285842605381
